In [ ]:
import os
import pandas as pd
import oracledb

def get_oracle_connection():
    host = os.getenv('ORACLE_HOST')
    port = os.getenv('ORACLE_PORT')
    service = os.getenv('ORACLE_SERVICE')
    user = os.getenv('ORACLE_USER')
    password = os.getenv('ORACLE_PASSWORD')

    dsn = f"{host}:{port}/{service}"
    return oracledb.connect(user=user, password=password, dsn=dsn)

def main():
    queries = [
        {
            "num": 1,
            "title": "Ingresos y Pedidos por Mes (Evolución Temporal)",
            "sql": """
                SELECT 
                    TO_CHAR(ORDER_DATE, 'YYYY-MM') AS MES,
                    COUNT(ORDER_ID) AS TOTAL_PEDIDOS,
                    ROUND(SUM(TOTAL_AMOUNT), 2) AS INGRESOS_TOTALES,
                    ROUND(AVG(TOTAL_AMOUNT), 2) AS TICKET_MEDIO
                FROM ORDERS
                WHERE STATUS != 'CANCELADO'
                GROUP BY TO_CHAR(ORDER_DATE, 'YYYY-MM')
                ORDER BY MES DESC
                FETCH FIRST 12 ROWS ONLY
            """
        },
        {
            "num": 2,
            "title": "Top 5 Productos Más Vendidos y su Rentabilidad",
            "sql": """
                SELECT 
                    p.PRODUCT_ID,
                    p.NAME AS PRODUCTO,
                    c.NAME AS CATEGORIA,
                    SUM(oi.QUANTITY) AS UNIDADES_VENDIDAS,
                    ROUND(SUM((oi.UNIT_PRICE * oi.QUANTITY) - oi.DISCOUNT), 2) AS FACTURACION_TOTAL,
                    ROUND(SUM((oi.UNIT_PRICE - p.COST) * oi.QUANTITY - oi.DISCOUNT), 2) AS BENEFICIO_ESTIMADO
                FROM ORDER_ITEMS oi
                JOIN PRODUCTS p ON oi.PRODUCT_ID = p.PRODUCT_ID
                JOIN CATEGORIES c ON p.CATEGORY_ID = c.CATEGORY_ID
                JOIN ORDERS o ON oi.ORDER_ID = o.ORDER_ID
                WHERE o.STATUS != 'CANCELADO'
                GROUP BY p.PRODUCT_ID, p.NAME, c.NAME
                ORDER BY UNIDADES_VENDIDAS DESC
                FETCH FIRST 5 ROWS ONLY
            """
        },
        {
            "num": 3,
            "title": "Tiempos Medios de Preparación y Entrega por País",
            "sql": """
                SELECT 
                    SEND_COUNTRY AS PAIS,
                    COUNT(ORDER_ID) AS PEDIDOS_ENTREGADOS,
                    ROUND(AVG(SEND_DATE - ORDER_DATE), 2) AS DIAS_ENVIO_MEDIO,
                    ROUND(AVG(RECEIVE_DATE - SEND_DATE), 2) AS DIAS_TRANSITO_MEDIO,
                    ROUND(AVG(RECEIVE_DATE - ORDER_DATE), 2) AS DIAS_TOTAL_ENTREGA
                FROM ORDERS
                WHERE STATUS = 'ENTREGADO' 
                  AND SEND_DATE IS NOT NULL 
                  AND RECEIVE_DATE IS NOT NULL
                GROUP BY SEND_COUNTRY
                ORDER BY PEDIDOS_ENTREGADOS DESC
            """
        },
        {
            "num": 4,
            "title": "Satisfacción de Cliente y Rating Medio por Categoría",
            "sql": """
                SELECT 
                    c.NAME AS CATEGORIA,
                    COUNT(r.REVIEW_ID) AS TOTAL_RESEÑAS,
                    ROUND(AVG(r.RATING), 2) AS RATING_PROMEDIO,
                    SUM(CASE WHEN r.RATING >= 4 THEN 1 ELSE 0 END) AS RESEÑAS_POSITIVAS,
                    SUM(CASE WHEN r.RATING <= 2 THEN 1 ELSE 0 END) AS RESEÑAS_NEGATIVAS
                FROM REVIEWS r
                JOIN PRODUCTS p ON r.PRODUCT_ID = p.PRODUCT_ID
                JOIN CATEGORIES c ON p.CATEGORY_ID = c.CATEGORY_ID
                GROUP BY c.NAME
                ORDER BY RATING_PROMEDIO DESC
            """
        },
        {
            "num": 5,
            "title": "Rendimiento de Adquisición de Clientes por Canal",
            "sql": """
                SELECT 
                    c.CANAL,
                    COUNT(DISTINCT c.CUSTOMER_ID) AS TOTAL_CLIENTES,
                    COUNT(DISTINCT o.ORDER_ID) AS TOTAL_PEDIDOS,
                    ROUND(SUM(o.TOTAL_AMOUNT), 2) AS FACTURACION_ACUMULADA,
                    ROUND(SUM(o.TOTAL_AMOUNT) / COUNT(DISTINCT c.CUSTOMER_ID), 2) AS VALOR_MEDIO_CLIENTE
                FROM CUSTOMERS c
                LEFT JOIN ORDERS o ON c.CUSTOMER_ID = o.CUSTOMER_ID AND o.STATUS != 'CANCELADO'
                GROUP BY c.CANAL
                ORDER BY FACTURACION_ACUMULADA DESC
            """
        }
    ]

    try:
        conn = get_oracle_connection()
        cursor = conn.cursor()
        print("Conexión exitosa a Oracle Database via oracledb.\n")

        for q in queries:
            print(f"{'='*70}")
            print(f"QUERY {q['num']}: {q['title']}")
            print(f"{'='*70}")

            cursor.execute(q['sql'])
            
            # Obtener nombres de columnas y tuplas de datos
            columns = [col[0] for col in cursor.description]
            rows = cursor.fetchall()

            df = pd.DataFrame(rows, columns=columns)

            if df.empty:
                print("⚠ La consulta no devolvió resultados.\n")
            else:
                print(df.to_string(index=False))
                print("\n")

        cursor.close()
        conn.close()

    except oracledb.Error as e:
        print(f"[ERROR Oracle]: {e}")
    except Exception as e:
        print(f"[ERROR inesperado]: {e}")

if __name__ == '__main__':
    main()